# Лабораторная 2. Формирование отчётов в Apache Spark

In [2]:
from pyspark.sql import SparkSession
from math import sqrt
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [3]:
# Создаем чистую сессию, которая сама всё настроит для локальной работы
spark = SparkSession.builder \
    .appName("MyMatrixLab") \
    .master("local[*]") \
    .getOrCreate()

# Вытаскиваем контекст из сессии для работы с RDD
sc = spark.sparkContext

print("Ура! Spark Context успешно запущен!")
print("Версия Spark:", sc.version)

Ура! Spark Context успешно запущен!
Версия Spark: 3.5.0


# Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года. 
Получившийся отчёт сохранить в формате Apache Parquet.

In [12]:
# Загружаем справочник языков
languages_df = spark.read.csv("programming-languages.csv", header=True)
valid_languages = languages_df.select(F.lower(F.col("name")).alias("lang_name"))


raw_posts = spark.read.text("posts_sample.xml")

# Вытаскиваем год и теги 
tags_raw = F.regexp_extract(F.col("value"), r'Tags="([^"]+)"', 1)
# Заменяем &lt; на < и &gt; на >
tags_unescaped = F.regexp_replace(F.regexp_replace(tags_raw, "&lt;", "<"), "&gt;", ">")

parsed_posts = raw_posts.select(
    F.regexp_extract(F.col("value"), r'CreationDate="(\d{4})', 1).cast("int").alias("year"),
    tags_unescaped.alias("tags")
)

# Оставляем только нужный период времени
filtered_posts = parsed_posts.filter(
    (F.col("year") >= 2010) & 
    (F.col("year") <= 2020) & 
    (F.col("tags") != "")
)

# Разбиваем строку тегов на отдельные ряды
clean_tags = F.split(F.regexp_replace(F.col("tags"), r'(^<|>$)', ''), r'><')
exploded_tags = filtered_posts.select("year", F.explode(clean_tags).alias("tag"))

# Оставляем только те теги, которые являются языками программирования
matched_tags = exploded_tags.join(valid_languages, exploded_tags.tag == valid_languages.lang_name, "inner")

# Считаем популярность языков по годам
tag_counts = matched_tags.groupBy("year", "tag").count()

window_spec = Window.partitionBy("year").orderBy(F.col("count").desc())
ranked_tags = tag_counts.withColumn("rank", F.rank().over(window_spec))

# Оставляем только первые 10 мест и красиво сортируем
top_10_report = ranked_tags.filter(F.col("rank") <= 10).orderBy("year", "rank")

top_10_report.show(120, truncate=False)

top_10_report.toPandas().to_parquet("top_10_languages_final.parquet")
print("Отчет сохранен")

+----+-----------+-----+----+
|year|tag        |count|rank|
+----+-----------+-----+----+
|2010|java       |52   |1   |
|2010|php        |46   |2   |
|2010|javascript |44   |3   |
|2010|python     |26   |4   |
|2010|objective-c|23   |5   |
|2010|c          |20   |6   |
|2010|ruby       |12   |7   |
|2010|delphi     |8    |8   |
|2010|applescript|3    |9   |
|2010|r          |3    |9   |
|2010|perl       |3    |9   |
|2010|bash       |3    |9   |
|2011|php        |102  |1   |
|2011|java       |93   |2   |
|2011|javascript |83   |3   |
|2011|python     |37   |4   |
|2011|objective-c|34   |5   |
|2011|c          |24   |6   |
|2011|ruby       |20   |7   |
|2011|perl       |9    |8   |
|2011|delphi     |8    |9   |
|2011|bash       |7    |10  |
|2012|php        |154  |1   |
|2012|javascript |132  |2   |
|2012|java       |124  |3   |
|2012|python     |69   |4   |
|2012|objective-c|45   |5   |
|2012|ruby       |27   |6   |
|2012|c          |27   |6   |
|2012|bash       |10   |8   |
|2012|r   